In [ ]:
import numpy as np
import plotly.graph_objects as go

# Mål på lokalet 

room_length = 8.80
room_width = 7.20
room_height = 2.92

professor_area_length = 1.60

number_of_rows = 8
seats_per_row = 8

row_depth = (
    room_length - professor_area_length
) / number_of_rows

aisle_width = 0.80

lowest_ear_height = 0.85
highest_ear_height = 1.22

floor_rise = (
    highest_ear_height
    - lowest_ear_height
)

seat_width = 0.50
seat_depth = 0.50

# Hældning af gulvet 

first_row_x = (
    professor_area_length
    + 0.5 * row_depth
)

last_row_x = (
    professor_area_length
    + (number_of_rows - 0.5) * row_depth
)


def floor_height(x_value, y_value):
    """
    Fladt gulv foran og i sidegangene.

    Det centrale sædeområde hælder jævnt
    op mod bagvæggen.
    """

    x_value = np.asarray(x_value)
    y_value = np.asarray(y_value)

    x_value, y_value = np.broadcast_arrays(
        x_value,
        y_value
    )

    height = np.zeros(
        x_value.shape,
        dtype=float
    )

    inside_seating_width = (
        (y_value >= aisle_width)
        & (
            y_value
            <= room_width - aisle_width
        )
    )

    behind_professor = (
        x_value >= professor_area_length
    )

    inside_seating_area = (
        inside_seating_width
        & behind_professor
    )

    slope_position = (
        x_value - first_row_x
    ) / (
        last_row_x - first_row_x
    )

    slope_position = np.clip(
        slope_position,
        0.0,
        1.0
    )

    height[inside_seating_area] = (
        floor_rise
        * slope_position[inside_seating_area]
    )

    return height

# Placering af sæder 

row_x_positions = np.linspace(
    first_row_x,
    last_row_x,
    number_of_rows
)

row_floor_heights = np.linspace(
    0.0,
    floor_rise,
    number_of_rows
)

seat_y_positions = np.linspace(
    aisle_width + 0.35,
    room_width - aisle_width - 0.35,
    seats_per_row
)

seat_positions = []

for row in range(number_of_rows):
    for seat_y in seat_y_positions:
        seat_positions.append([
            row_x_positions[row],
            seat_y,
            row_floor_heights[row]
        ])

seat_positions = np.asarray(
    seat_positions
)

# Professor position 

professor_position = np.array([
    0.80,
    room_width / 2,
    1.65
])

# Figur 

room_figure = go.Figure()


# Gulvet - med hældning

floor_x = np.linspace(
    0,
    room_length,
    150
)

floor_y = np.linspace(
    0,
    room_width,
    120
)

Floor_X, Floor_Y = np.meshgrid(
    floor_x,
    floor_y,
    indexing="ij"
)

Floor_Z = floor_height(
    Floor_X,
    Floor_Y
)

room_figure.add_trace(
    go.Surface(
        x=Floor_X,
        y=Floor_Y,
        z=Floor_Z,

        surfacecolor=np.zeros_like(
            Floor_Z
        ),

        colorscale=[
            [0, "rgb(166, 112, 66)"],
            [1, "rgb(166, 112, 66)"]
        ],

        showscale=False,
        opacity=0.90,

        name="Trægulv",
        showlegend=True
    )
)

# Murstensvæg med tavle der kommer senere

front_wall_y = np.linspace(
    0,
    room_width,
    40
)

front_wall_z = np.linspace(
    0,
    room_height,
    30
)

Front_Y, Front_Z = np.meshgrid(
    front_wall_y,
    front_wall_z
)

Front_X = np.zeros_like(
    Front_Y
)

room_figure.add_trace(
    go.Surface(
        x=Front_X,
        y=Front_Y,
        z=Front_Z,

        surfacecolor=np.zeros_like(
            Front_X
        ),

        colorscale=[
            [0, "rgb(150, 85, 65)"],
            [1, "rgb(150, 85, 65)"]
        ],

        showscale=False,
        opacity=0.90,

        name="Forvæg",
        showlegend=True
    )
)

# Tavlen med væg rundt om 

wall_border = 0.40

chalkboard_y_min = 0.0

chalkboard_y_max = (
    room_width - wall_border
)

chalkboard_z_min = wall_border

chalkboard_z_max = (
    room_height - wall_border
)

board_y = np.linspace(
    chalkboard_y_min,
    chalkboard_y_max,
    40
)

board_z = np.linspace(
    chalkboard_z_min,
    chalkboard_z_max,
    25
)

Board_Y, Board_Z = np.meshgrid(
    board_y,
    board_z
)

# Tavlen flyttes 5 mm ind i lokalet, så Plotly viser den foran murstensvæggen

Board_X = np.full_like(
    Board_Y,
    0.005
)

room_figure.add_trace(
    go.Surface(
        x=Board_X,
        y=Board_Y,
        z=Board_Z,

        surfacecolor=np.zeros_like(
            Board_X
        ),

        colorscale=[
            [0, "rgb(25, 75, 52)"],
            [1, "rgb(25, 75, 52)"]
        ],

        showscale=False,
        opacity=1.0,

        name="Tavle",
        showlegend=True
    )
)

# Væg med vinduer 

window_x = np.linspace(
    0,
    room_length,
    50
)

window_z = np.linspace(
    0,
    room_height,
    25
)

Window_X, Window_Z = np.meshgrid(
    window_x,
    window_z
)

Window_Y = np.zeros_like(
    Window_X,
)

room_figure.add_trace(
    go.Surface(
        x=Window_X,
        y=Window_Y,
        z=Window_Z,

        surfacecolor=np.zeros_like(
            Window_X
        ),

        colorscale=[
            [0, "rgb(120, 190, 220)"],
            [1, "rgb(120, 190, 220)"]
        ],

        showscale=False,
        opacity=0.22,

        name="Vinduesvæg",
        showlegend=True
    )
)

# Venstre væg

left_wall_x = np.linspace(
    0,
    room_length,
    50
)

left_wall_z = np.linspace(
    0,
    room_height,
    25
)

Left_X, Left_Z = np.meshgrid(
    left_wall_x,
    left_wall_z
)

Left_Y = np.full_like(
    Left_X, 
    room_width
)

room_figure.add_trace(
    go.Surface(
        x=Left_X,
        y=Left_Y,
        z=Left_Z,

        surfacecolor=np.zeros_like(
            Left_X
        ),

        colorscale=[
            [0, "rgb(150, 85, 65)"],
            [1, "rgb(150, 85, 65)"]
        ],

        showscale=False,
        opacity=0.18,

        name="Venstre murstensvæg",
        showlegend=True
    )
)

# Bagvæg 

back_wall_y = np.linspace(
    0,
    room_width,
    40
)

back_wall_z = np.linspace(
    0,
    room_height,
    25
)

Back_Y, Back_Z = np.meshgrid(
    back_wall_y,
    back_wall_z
)

Back_X = np.full_like(
    Back_Y,
    room_length
)

room_figure.add_trace(
    go.Surface(
        x=Back_X,
        y=Back_Y,
        z=Back_Z,

        surfacecolor=np.zeros_like(
            Back_X
        ),

        colorscale=[
            [0, "rgb(150, 85, 65)"],
            [1, "rgb(150, 85, 65)"]
        ],

        showscale=False,
        opacity=0.18,

        name="Bagvæg",
        showlegend=False
    )
)

# Loft - gennemsigtigt 

ceiling_x = np.linspace(
    0,
    room_length,
    50
)

ceiling_y = np.linspace(
    0,
    room_width,
    40
)

Ceiling_X, Ceiling_Y = np.meshgrid(
    ceiling_x,
    ceiling_y
)

Ceiling_Z = np.full_like(
    Ceiling_X,
    room_height
)

room_figure.add_trace(
    go.Surface(
        x=Ceiling_X,
        y=Ceiling_Y,
        z=Ceiling_Z,

        surfacecolor=np.zeros_like(
            Ceiling_X
        ),

        colorscale=[
            [0, "rgb(188, 145, 95)"],
            [1, "rgb(188, 145, 95)"]
        ],

        showscale=False,
        opacity=0.10,

        name="Bræddeloft",
        showlegend=True
    )
)

# Funktion til boxsæder

vertices = []

triangles_i = []
triangles_j = []
triangles_k = []


def add_box(
    center_x,
    center_y,
    bottom_z,
    size_x,
    size_y,
    size_z
):
    start_index = len(vertices)

    x_min = center_x - size_x / 2
    x_max = center_x + size_x / 2

    y_min = center_y - size_y / 2
    y_max = center_y + size_y / 2

    z_min = bottom_z
    z_max = bottom_z + size_z

    vertices.extend([
        [x_min, y_min, z_min],
        [x_max, y_min, z_min],
        [x_max, y_max, z_min],
        [x_min, y_max, z_min],

        [x_min, y_min, z_max],
        [x_max, y_min, z_max],
        [x_max, y_max, z_max],
        [x_min, y_max, z_max]
    ])

    faces = [
        (0, 1, 2),
        (0, 2, 3),

        (4, 6, 5),
        (4, 7, 6),

        (0, 4, 5),
        (0, 5, 1),

        (1, 5, 6),
        (1, 6, 2),

        (2, 6, 7),
        (2, 7, 3),

        (3, 7, 4),
        (3, 4, 0)
    ]

    for face in faces:
        triangles_i.append(
            start_index + face[0]
        )

        triangles_j.append(
            start_index + face[1]
        )

        triangles_k.append(
            start_index + face[2]
        )

# Polstrede sæder

for seat_x, seat_y, seat_floor_z in seat_positions:

    # Siddeflade
    add_box(
        center_x=seat_x,
        center_y=seat_y,
        bottom_z=seat_floor_z + 0.32,
        size_x=0.45,
        size_y=0.50,
        size_z=0.12
    )

    # Ryglæn
    add_box(
        center_x=seat_x + 0.18,
        center_y=seat_y,
        bottom_z=seat_floor_z + 0.40,
        size_x=0.10,
        size_y=0.50,
        size_z=0.45
    )


vertices = np.asarray(
    vertices
)

room_figure.add_trace(
    go.Mesh3d(
        x=vertices[:, 0],
        y=vertices[:, 1],
        z=vertices[:, 2],

        i=triangles_i,
        j=triangles_j,
        k=triangles_k,

        color="rgb(180, 35, 45)",
        opacity=0.90,

        flatshading=True,

        name="Polstrede sæder",
        showlegend=True
    )
)

# 15. Mathias 

room_figure.add_trace(
    go.Scatter3d(
        x=[professor_position[0]],
        y=[professor_position[1]],
        z=[professor_position[2]],

        mode="markers+text",

        marker=dict(
            size=8,
            color="mediumorchid",
            symbol="circle"
        ),

        text=["Mathias"],
        textposition="top center",

        name="Mathias"
    )
)

# Figur

room_figure.update_layout(
    title="3D-model af auditoriet",

    scene=dict(
        xaxis=dict(
            title="Længde [m]",
            range=[0, room_length]
        ),

        yaxis=dict(
            title="Bredde [m]",
            range=[0, room_width]
        ),

        zaxis=dict(
            title="Højde [m]",
            range=[0, room_height]
        ),

        aspectmode="manual",

        aspectratio=dict(
            x=room_length / room_width,
            y=1,
            z=room_height / room_width
        ),

        camera=dict(
            eye=dict(
                x=1.6,
                y=-1.8,
                z=1.1
            )
        )
    ),

    width=1100,
    height=750,

    margin=dict(
        l=0,
        r=0,
        b=0,
        t=60
    )
)
    
room_figure.show()